In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
#SET WORKING DIRECTORY TO PROJECT FOLDER IN GOOGLE DRIVE

import os

project_path = "/content/drive/MyDrive/trademarkia_project"

os.chdir(project_path)

print("Current working directory set to:")
print(os.getcwd())

Current working directory set to:
/content/drive/MyDrive/trademarkia_project


In [ ]:
# 1. IMPORT LIBRARIES

import os
import re
import zipfile
import pandas as pd
from pathlib import Path

In [ ]:
# 2. VERIFY DATASET FILE IN /content

content_files = os.listdir("/content")
print("Files in /content:")
print(content_files)

Files in /content:
['.config', 'twenty+newsgroups.zip', '.ipynb_checkpoints']


In [ ]:
# 3. EXTRACT DATASET ZIP FILE

zip_path = "/content/twenty+newsgroups.zip"
extract_path = "/content/twenty_newsgroups"

# Create extraction directory if it does not exist
os.makedirs(extract_path, exist_ok=True)

# Extract the zip file
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully.")
print("Extracted to:", extract_path)

Dataset extracted successfully.
Extracted to: /content/twenty_newsgroups


In [ ]:
# 4. INSPECT EXTRACTED DATASET STRUCTURE

top_level_items = os.listdir(extract_path)
print("Top-level items inside extracted folder:")
print(top_level_items[:20])

Top-level items inside extracted folder:
['20newsgroups.html', 'mini_newsgroups.tar.gz', '20newsgroups.data.html', '20_newsgroups.tar.gz']


In [ ]:
# 5.Find the folder that contains category directories

import os

base = "/content/twenty_newsgroups"

for item in os.listdir(base):
    item_path = os.path.join(base, item)

    if os.path.isdir(item_path):
        subfolders = os.listdir(item_path)

        if len(subfolders) >= 10:
            dataset_root = item_path
            break
else:
    dataset_root = base

print("Dataset root:", dataset_root)

Dataset root: /content/twenty_newsgroups


In [ ]:
os.listdir(dataset_root)

['20newsgroups.html',
 'mini_newsgroups.tar.gz',
 '20newsgroups.data.html',
 '20_newsgroups.tar.gz']

In [ ]:
# 6. EXTRACT THE ACTUAL DATASET FROM .tar.gz

import tarfile
import os

tar_path = os.path.join(dataset_root, "20_newsgroups.tar.gz")
final_extract_path = "/content/20_newsgroups"

os.makedirs(final_extract_path, exist_ok=True)

with tarfile.open(tar_path, "r:gz") as tar:
    tar.extractall(path=final_extract_path)

print("Actual dataset extracted successfully.")
print(os.listdir(final_extract_path))

/tmp/ipykernel_422/1135258203.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=final_extract_path)


Actual dataset extracted successfully.
['20_newsgroups']


In [ ]:
# 7. VERIFY CATEGORY FOLDERS

dataset_root = "/content/20_newsgroups/20_newsgroups"

categories = os.listdir(dataset_root)

print("Number of categories:", len(categories))
print(categories)

Number of categories: 20
['comp.sys.mac.hardware', 'sci.med', 'comp.os.ms-windows.misc', 'rec.sport.baseball', 'rec.autos', 'rec.sport.hockey', 'sci.space', 'talk.religion.misc', 'soc.religion.christian', 'comp.sys.ibm.pc.hardware', 'comp.windows.x', 'sci.crypt', 'talk.politics.mideast', 'talk.politics.misc', 'rec.motorcycles', 'misc.forsale', 'comp.graphics', 'sci.electronics', 'alt.atheism', 'talk.politics.guns']


In [ ]:
# 8. LOAD ALL DOCUMENTS INTO A DATAFRAME

import os
import pandas as pd

data = []
doc_id = 0

for category in categories:
    category_path = os.path.join(dataset_root, category)

    for filename in os.listdir(category_path):
        file_path = os.path.join(category_path, filename)

        if not os.path.isfile(file_path):
            continue

        try:
            with open(file_path, "r", encoding="latin1") as f:
                text = f.read()

            data.append({
                "doc_id": doc_id,
                "category": category,
                "filename": filename,
                "file_path": file_path,
                "raw_text": text
            })

            doc_id += 1

        except Exception as e:
            print(f"Could not read file: {file_path}")
            print("Error:", e)

df = pd.DataFrame(data)

print("Total documents loaded:", len(df))
df.head()

Total documents loaded: 19997


,doc_id,category,filename,file_path,raw_text
0,0,comp.sys.mac.hardware,52246,/content/20_newsgroups/20_newsgroups/comp.sys....,Newsgroups: comp.sys.mac.hardware\nPath: canta...
1,1,comp.sys.mac.hardware,52309,/content/20_newsgroups/20_newsgroups/comp.sys....,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...
2,2,comp.sys.mac.hardware,50467,/content/20_newsgroups/20_newsgroups/comp.sys....,Newsgroups: comp.sys.mac.hardware\nPath: canta...
3,3,comp.sys.mac.hardware,51640,/content/20_newsgroups/20_newsgroups/comp.sys....,Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv....
4,4,comp.sys.mac.hardware,50537,/content/20_newsgroups/20_newsgroups/comp.sys....,Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv....


In [ ]:
# 9. INSPECT DATASET STRUCTURE

print("DataFrame shape:", df.shape)

print("\nColumns in dataset:")
print(df.columns)

print("\nCategory distribution:")
print(df["category"].value_counts())

DataFrame shape: (19997, 5)

Columns in dataset:
Index(['doc_id', 'category', 'filename', 'file_path', 'raw_text'], dtype='object')

Category distribution:
category
comp.sys.mac.hardware       1000
sci.med                     1000
comp.os.ms-windows.misc     1000
rec.sport.baseball          1000
rec.autos                   1000
rec.sport.hockey            1000
sci.space                   1000
talk.religion.misc          1000
comp.sys.ibm.pc.hardware    1000
comp.windows.x              1000
sci.crypt                   1000
talk.politics.mideast       1000
comp.graphics               1000
talk.politics.misc          1000
rec.motorcycles             1000
misc.forsale                1000
alt.atheism                 1000
sci.electronics             1000
talk.politics.guns          1000
soc.religion.christian       997
Name: count, dtype: int64


In [ ]:
# 10. INSPECT A SAMPLE RAW DOCUMENT

sample_index = 0

print("Category:", df.loc[sample_index, "category"])
print("Filename:", df.loc[sample_index, "filename"])

print("\nPreview of raw document:\n")
print(df.loc[sample_index, "raw_text"][:1500])

Category: comp.sys.mac.hardware
Filename: 52246

Preview of raw document:

Newsgroups: comp.sys.mac.hardware
Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!uwm.edu!ux1.cso.uiuc.edu!newsrelay.iastate.edu!news.iastate.edu!kmradke
From: kmradke@iastate.edu (Kevin M Radke)
Subject: Unknown Mac board NATIONAL INSTRUMENTS NB-DMA-8
Message-ID: <C5yGDq.6MI@news.iastate.edu>
Sender: news@news.iastate.edu (USENET News System)
Organization: Iowa State University, Ames, IA
Distribution: usa
Date: Fri, 23 Apr 1993 21:15:19 GMT
Lines: 22

I need help identifying this board that I found stuffed away in a corner.

As the title says, all that is printed on it is NATIONAL INSTRUMENTS NB-DMA-8.
It fits fine in my Mac IIci and snooper gives the very same name for the
board.  It looks like it has an HP-IB connector on the back of it and
another connector on the top (2 rows by 25 pins).  It also looks like
it has an Intel processor on 

In [ ]:
# 11. DEFINE TEXT CLEANING FUNCTION

import re

def clean_newsgroup_text(text):

    if not isinstance(text, str):
        return ""

    # remove common email/news headers
    text = re.sub(r'(?im)^(from|subject|organization|lines|distribution|keywords|date|path|xref|message-id|sender):.*$', '', text)

    # remove quoted replies (lines starting with >)
    text = re.sub(r'(?m)^>.*$', '', text)

    # remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # remove long separators
    text = re.sub(r'[-_=]{2,}', ' ', text)

    # remove strange symbols
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?;:()\'"-]', ' ', text)

    # remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# 12. APPLY TEXT CLEANING TO THE DATASET

df["clean_text"] = df["raw_text"].apply(clean_newsgroup_text)

print("Text cleaning applied successfully.")

df[["doc_id", "category", "clean_text"]].head()

Text cleaning applied successfully.


,doc_id,category,clean_text
0,0,comp.sys.mac.hardware,Newsgroups: comp.sys.mac.hardware I need help ...
1,1,comp.sys.mac.hardware,Newsgroups: comp.sys.mac.hardware Reply-To: (T...
2,2,comp.sys.mac.hardware,Newsgroups: comp.sys.mac.hardware Followup-To:...
3,3,comp.sys.mac.hardware,Newsgroups: comp.sys.mac.hardware NNTP-Posting...
4,4,comp.sys.mac.hardware,Newsgroups: comp.sys.mac.hardware NNTP-Posting...


In [ ]:
# 13. UPDATE THE TEXT CLEANING FUNCTION

import re

def clean_newsgroup_text(text):

    if not isinstance(text, str):
        return ""

    # remove common newsgroup/email header lines
    text = re.sub(
        r'(?im)^(from|subject|organization|lines|distribution|keywords|date|path|xref|message-id|sender|newsgroups|reply-to|followup-to|nntp-posting-host|article-i\.d\.|summary|expires):.*$',
        '',
        text
    )

    # remove quoted replies
    text = re.sub(r'(?m)^>.*$', '', text)

    # remove email addresses
    text = re.sub(r'\S+@\S+', ' ', text)

    # remove URLs
    text = re.sub(r'http\S+|www\S+', ' ', text)

    # remove separator lines
    text = re.sub(r'[-_=]{2,}', ' ', text)

    # keep only useful characters
    text = re.sub(r'[^a-zA-Z0-9\s\.,!?;:()\'"-]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
# 14. RE-APPLY THE UPDATED CLEANING FUNCTION

df["clean_text"] = df["raw_text"].apply(clean_newsgroup_text)

print("Updated text cleaning applied successfully.")

df[["doc_id", "category", "clean_text"]].head()

Updated text cleaning applied successfully.


,doc_id,category,clean_text
0,0,comp.sys.mac.hardware,I need help identifying this board that I foun...
1,1,comp.sys.mac.hardware,For sale: Mac IIfx with the following config 2...
2,2,comp.sys.mac.hardware,References: In article (Robert Mclean) wrote: ...
3,3,comp.sys.mac.hardware,I have a new 25 MHz Motorola 68040 that I am w...
4,4,comp.sys.mac.hardware,In article (Thumper) writes: I think the origi...


In [ ]:
# 15. COMPARE RAW TEXT AND CLEANED TEXT

sample_index = 0

print("RAW TEXT:\n")
print(df.loc[sample_index, "raw_text"][:1500])

print("\n" + "-" * 80 + "\n")

print("CLEANED TEXT:\n")
print(df.loc[sample_index, "clean_text"][:1500])

RAW TEXT:

Newsgroups: comp.sys.mac.hardware
Path: cantaloupe.srv.cs.cmu.edu!magnesium.club.cc.cmu.edu!news.sei.cmu.edu!cis.ohio-state.edu!zaphod.mps.ohio-state.edu!uwm.edu!ux1.cso.uiuc.edu!newsrelay.iastate.edu!news.iastate.edu!kmradke
From: kmradke@iastate.edu (Kevin M Radke)
Subject: Unknown Mac board NATIONAL INSTRUMENTS NB-DMA-8
Message-ID: <C5yGDq.6MI@news.iastate.edu>
Sender: news@news.iastate.edu (USENET News System)
Organization: Iowa State University, Ames, IA
Distribution: usa
Date: Fri, 23 Apr 1993 21:15:19 GMT
Lines: 22

I need help identifying this board that I found stuffed away in a corner.

As the title says, all that is printed on it is NATIONAL INSTRUMENTS NB-DMA-8.
It fits fine in my Mac IIci and snooper gives the very same name for the
board.  It looks like it has an HP-IB connector on the back of it and
another connector on the top (2 rows by 25 pins).  It also looks like
it has an Intel processor on it (#A82380-16 Intel '86)

On an EEPROM there is a sticker with 

In [ ]:
# 16. REMOVE VERY SHORT DOCUMENTS

df["clean_length"] = df["clean_text"].apply(lambda x: len(x.split()))

# keep documents with at least 10 words
df = df[df["clean_length"] >= 10].reset_index(drop=True)

print("Documents remaining after filtering:", len(df))
df.head()

Documents remaining after filtering: 19812


,doc_id,category,filename,file_path,raw_text,clean_text,clean_length
0,0,comp.sys.mac.hardware,52246,/content/20_newsgroups/20_newsgroups/comp.sys....,Newsgroups: comp.sys.mac.hardware\nPath: canta...,I need help identifying this board that I foun...,143
1,1,comp.sys.mac.hardware,52309,/content/20_newsgroups/20_newsgroups/comp.sys....,Path: cantaloupe.srv.cs.cmu.edu!magnesium.club...,For sale: Mac IIfx with the following config 2...,40
2,2,comp.sys.mac.hardware,50467,/content/20_newsgroups/20_newsgroups/comp.sys....,Newsgroups: comp.sys.mac.hardware\nPath: canta...,References: In article (Robert Mclean) wrote: ...,131
3,3,comp.sys.mac.hardware,51640,/content/20_newsgroups/20_newsgroups/comp.sys....,Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv....,I have a new 25 MHz Motorola 68040 that I am w...,89
4,4,comp.sys.mac.hardware,50537,/content/20_newsgroups/20_newsgroups/comp.sys....,Path: cantaloupe.srv.cs.cmu.edu!crabapple.srv....,In article (Thumper) writes: I think the origi...,97


In [ ]:
# 17. SAVE THE CLEANED DATASET

output_path = "/content/cleaned_20newsgroups.csv"

df.to_csv(output_path, index=False)

print("Cleaned dataset saved successfully.")
print("File location:", output_path)

Cleaned dataset saved successfully.
File location: /content/cleaned_20newsgroups.csv


In [ ]:
# Fast move of all files from /content to Drive project folder

import os

project_root = "/content/drive/MyDrive/trademarkia_project"
os.makedirs(project_root, exist_ok=True)

!mv /content/* "$project_root" 2>/dev/null

# return to project folder so all future files save there
os.chdir(project_root)

print("Moved files to:", project_root)
print("Current working directory:", os.getcwd())
print("Files now in project folder:")
print(os.listdir(project_root))

Moved files to: /content/drive/MyDrive/trademarkia_project
Current working directory: /content/drive/MyDrive/trademarkia_project
Files now in project folder:
['.config', 'twenty+newsgroups.zip', '.ipynb_checkpoints', '20_newsgroups', 'cleaned_20newsgroups.csv', 'drive', 'twenty_newsgroups']


In [ ]:
# 18. INSTALL REQUIRED LIBRARIES

!pip install sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.3 MB/s eta 0:00:00


In [ ]:
# 19. LOAD SENTENCE TRANSFORMER MODEL

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


In [ ]:
# 20. GENERATE EMBEDDINGS FOR DOCUMENTS

texts = df["clean_text"].tolist()

embeddings = embedding_model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True
)

print("Embeddings generated.")
print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/310 [00:00<?, ?it/s]

Embeddings generated.
Embedding shape: (19812, 384)


In [ ]:
# 21. CREATE FAISS VECTOR INDEX

import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS index created successfully.")
print("Total vectors stored in index:", index.ntotal)

FAISS index created successfully.
Total vectors stored in index: 19812


In [ ]:
# 22. TEST SEMANTIC SEARCH

query = "space shuttle launch"

query_vector = embedding_model.encode([query])

k = 5  # number of results to retrieve

distances, indices = index.search(query_vector, k)

print("Query:", query)
print("\nTop results:\n")

for i, idx in enumerate(indices[0]):
    print(f"Result {i+1}")
    print("Category:", df.iloc[idx]["category"])
    print("Text:", df.iloc[idx]["clean_text"][:300])
    print()

Query: space shuttle launch

Top results:

Result 1
Category: sci.space
Text: References: Well, you better not get the shuttle as your launch vehicle. and most ELV's have too far of a backlog for political messages. If during the campaign season, the candidates for president had launched one, right around now we'd be getting a launch for PEROT 92. and if they had used the shu

Result 2
Category: sci.space
Text: Supersedes: Approved: References: Archive-name: space schedule Last-modified: Date: 93 04 01 14:39:23 SPACE SHUTTLE ANSWERS, LAUNCH SCHEDULES, TV COVERAGE SHUTTLE LAUNCHINGS AND LANDINGS; SCHEDULES AND HOW TO SEE THEM Shuttle operations are discussed in the Usenet group sci.space.shuttle, and Ken Ho

Result 3
Category: sci.space
Text: X-Added: Forwarded by Space Digest Original-Sender: Approved: bboard-news gateway For an essay, I am writing about the space shuttle and a need for a better propulsion system. Through research, I have found that it is rather clumsy (i.e. all the ch

In [ ]:
# 23. SAVE THE FAISS INDEX

import faiss

faiss.write_index(index, "faiss_index.bin")

print("FAISS index saved successfully.")

FAISS index saved successfully.


In [ ]:
# 24. SAVE DOCUMENT EMBEDDINGS

import numpy as np

np.save("document_embeddings.npy", embeddings)

print("Embeddings saved successfully.")

Embeddings saved successfully.


In [ ]:
# 25. SAVE THE PROCESSED DATASET

df.to_csv("processed_documents.csv", index=False)

print("Processed dataset saved successfully.")

Processed dataset saved successfully.


In [ ]:
# 26. INSTALL LIBRARY FOR FUZZY CLUSTERING

!pip install scikit-fuzzy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 920.8/920.8 kB 20.2 MB/s eta 0:00:00


In [ ]:
# 27. IMPORT FUZZY CLUSTERING LIBRARY

import skfuzzy as fuzz
import numpy as np

print("Fuzzy clustering library imported successfully.")

Fuzzy clustering library imported successfully.


In [ ]:
# 28. PREPARE EMBEDDINGS FOR FUZZY CLUSTERING

embedding_matrix = embeddings.T

print("Embedding matrix prepared.")
print("Shape:", embedding_matrix.shape)

Embedding matrix prepared.
Shape: (384, 19812)


In [ ]:
# 29. RUN FUZZY C-MEANS CLUSTERING

n_clusters = 20

cntr, u, u0, d, jm, p, fpc = fuzz.cluster.cmeans(
    embedding_matrix,
    c=n_clusters,
    m=2,
    error=0.005,
    maxiter=1000,
    init=None
)

print("Fuzzy clustering completed.")
print("Number of clusters:", n_clusters)
print("Membership matrix shape:", u.shape)

Fuzzy clustering completed.
Number of clusters: 20
Membership matrix shape: (20, 19812)


In [ ]:
# 30. ASSIGN DOMINANT CLUSTER TO EACH DOCUMENT

dominant_clusters = np.argmax(u, axis=0)

df["dominant_cluster"] = dominant_clusters

print("Dominant cluster assigned to each document.")

df[["doc_id", "category", "dominant_cluster"]].head()

Dominant cluster assigned to each document.


,doc_id,category,dominant_cluster
0,0,comp.sys.mac.hardware,19
1,1,comp.sys.mac.hardware,11
2,2,comp.sys.mac.hardware,11
3,3,comp.sys.mac.hardware,11
4,4,comp.sys.mac.hardware,19


In [ ]:
# 31. SAVE CLUSTERING RESULTS

df.to_csv("documents_with_clusters.csv", index=False)

print("Dataset with cluster assignments saved successfully.")

Dataset with cluster assignments saved successfully.


In [ ]:
# 32. SAVE FUZZY CLUSTER MEMBERSHIP MATRIX

import numpy as np

np.save("cluster_membership_matrix.npy", u)

print("Cluster membership matrix saved successfully.")
print("Matrix shape:", u.shape)

Cluster membership matrix saved successfully.
Matrix shape: (20, 19812)


In [ ]:
# 33. SAVE CLUSTER CENTERS

import numpy as np

np.save("cluster_centers.npy", cntr)

print("Cluster centers saved successfully.")
print("Shape:", cntr.shape)

Cluster centers saved successfully.
Shape: (20, 384)


In [ ]:
# 34. INITIALIZE SEMANTIC CACHE

semantic_cache = {}

cache_stats = {
    "total_entries": 0,
    "hit_count": 0,
    "miss_count": 0
}

print("Semantic cache initialized.")
print("Cache entries:", len(semantic_cache))

Semantic cache initialized.
Cache entries: 0


In [ ]:
# 35. DEFINE COSINE SIMILARITY FUNCTION

from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]

print("Similarity function ready.")

Similarity function ready.


In [ ]:
# 36. SET CACHE SIMILARITY THRESHOLD

SIMILARITY_THRESHOLD = 0.85

print("Cache similarity threshold set to:", SIMILARITY_THRESHOLD)

Cache similarity threshold set to: 0.85


In [ ]:
# 37. DEFINE FUNCTION TO CHECK SEMANTIC CACHE

def check_semantic_cache(query, query_embedding):

    best_match = None
    best_score = 0

    for cached_query, cache_entry in semantic_cache.items():

        cached_embedding = cache_entry["embedding"]

        score = compute_similarity(query_embedding, cached_embedding)

        if score > best_score:
            best_score = score
            best_match = cached_query

    if best_score >= SIMILARITY_THRESHOLD:
        cache_stats["hit_count"] += 1

        return {
            "cache_hit": True,
            "matched_query": best_match,
            "similarity_score": best_score,
            "result": semantic_cache[best_match]["result"],
            "dominant_cluster": semantic_cache[best_match]["cluster"]
        }

    cache_stats["miss_count"] += 1
    return {"cache_hit": False}

In [ ]:
# 38. DEFINE FUNCTION TO STORE RESULTS IN CACHE

def store_in_cache(query, query_embedding, result_text, dominant_cluster):

    semantic_cache[query] = {
        "embedding": query_embedding,
        "result": result_text,
        "cluster": dominant_cluster
    }

    cache_stats["total_entries"] = len(semantic_cache)

    print("Query stored in semantic cache.")

In [ ]:
# 39. DEFINE QUERY PROCESSING FUNCTION

def process_query(query):

    # embed the query
    query_embedding = embedding_model.encode(query)

    # check semantic cache
    cache_result = check_semantic_cache(query, query_embedding)

    if cache_result["cache_hit"]:
        return cache_result

    # perform vector search
    query_vector = np.array([query_embedding])
    distances, indices = index.search(query_vector, 1)

    doc_index = indices[0][0]

    result_text = df.iloc[doc_index]["clean_text"]
    dominant_cluster = int(df.iloc[doc_index]["dominant_cluster"])

    # store result in cache
    store_in_cache(query, query_embedding, result_text, dominant_cluster)

    return {
        "query": query,
        "cache_hit": False,
        "similarity_score": None,
        "matched_query": None,
        "result": result_text,
        "dominant_cluster": dominant_cluster
    }

In [ ]:
# 40. TEST THE SEMANTIC CACHE

query = "space shuttle launch"

response = process_query(query)

print("Response:")
print(response)

Query stored in semantic cache.
Response:
{'query': 'space shuttle launch', 'cache_hit': False, 'similarity_score': None, 'matched_query': None, 'result': "References: Well, you better not get the shuttle as your launch vehicle. and most ELV's have too far of a backlog for political messages. If during the campaign season, the candidates for president had launched one, right around now we'd be getting a launch for PEROT 92. and if they had used the shuttle, we'd be seeing launches for NIXON now more then ever. pat", 'dominant_cluster': 4}


In [ ]:
# 41. TEST CACHE HIT

query = "space shuttle launch"

response = process_query(query)

print("Response:")
print(response)

Response:
{'cache_hit': True, 'matched_query': 'space shuttle launch', 'similarity_score': np.float32(1.0), 'result': "References: Well, you better not get the shuttle as your launch vehicle. and most ELV's have too far of a backlog for political messages. If during the campaign season, the candidates for president had launched one, right around now we'd be getting a launch for PEROT 92. and if they had used the shuttle, we'd be seeing launches for NIXON now more then ever. pat", 'dominant_cluster': 4}


In [ ]:
# 42. TEST SEMANTICALLY SIMILAR QUERY

query = "launch of the space shuttle"

response = process_query(query)

print("Response:")
print(response)

Response:
{'cache_hit': True, 'matched_query': 'space shuttle launch', 'similarity_score': np.float32(0.960251), 'result': "References: Well, you better not get the shuttle as your launch vehicle. and most ELV's have too far of a backlog for political messages. If during the campaign season, the candidates for president had launched one, right around now we'd be getting a launch for PEROT 92. and if they had used the shuttle, we'd be seeing launches for NIXON now more then ever. pat", 'dominant_cluster': 4}


In [ ]:
# 43. FUNCTION TO RETURN CACHE STATS

def get_cache_stats():

    total = cache_stats["hit_count"] + cache_stats["miss_count"]

    hit_rate = cache_stats["hit_count"] / total if total > 0 else 0

    return {
        "total_entries": cache_stats["total_entries"],
        "hit_count": cache_stats["hit_count"],
        "miss_count": cache_stats["miss_count"],
        "hit_rate": hit_rate
    }

print("Cache statistics function ready.")

Cache statistics function ready.


In [ ]:
# 44. FUNCTION TO CLEAR THE CACHE

def clear_cache():

    semantic_cache.clear()

    cache_stats["total_entries"] = 0
    cache_stats["hit_count"] = 0
    cache_stats["miss_count"] = 0

    print("Semantic cache cleared.")

In [ ]:
!pip install fastapi uvicorn

In [ ]:
# 46. IMPORT FASTAPI

from fastapi import FastAPI
from pydantic import BaseModel

print("FastAPI imported successfully.")

FastAPI imported successfully.


In [ ]:
# 47. CREATE FASTAPI APPLICATION

app = FastAPI()

print("FastAPI app initialized.")

FastAPI app initialized.


In [ ]:
# 48. DEFINE QUERY REQUEST MODEL

class QueryRequest(BaseModel):
    query: str

print("Query request model created.")

Query request model created.


In [ ]:
# 49. CREATE /query ENDPOINT

from fastapi import Body

@app.post("/query")
def query_endpoint(request: QueryRequest = Body(...)):

    query_text = request.query
    response = process_query(query_text)

    return response

In [ ]:
# 50. CREATE /cache/stats ENDPOINT

@app.get("/cache/stats")
def cache_stats_endpoint():
    return get_cache_stats()

In [ ]:
# 51. CREATE DELETE /cache ENDPOINT

@app.delete("/cache")
def clear_cache_endpoint():

    clear_cache()

    return {"message": "Cache cleared successfully"}

In [8]:
# 52. WRITE IMPROVED PRODUCTION-LIKE FASTAPI APPLICATION TO app.py

%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Create FastAPI application
app = FastAPI(title="Trademarkia Semantic Search API")

# Define request schema for POST /query
class QueryRequest(BaseModel):
    query: str

# Load saved project artifacts
df = pd.read_csv("documents_with_clusters.csv")
index = faiss.read_index("faiss_index.bin")
cluster_centers = np.load("cluster_centers.npy")
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize in-memory semantic cache and statistics
semantic_cache = {}
cache_stats = {
    "total_entries": 0,
    "hit_count": 0,
    "miss_count": 0
}

# Similarity threshold for deciding cache hit
SIMILARITY_THRESHOLD = 0.85

# Compute cosine similarity between two vectors
def compute_similarity(vec1, vec2):
    return cosine_similarity([vec1], [vec2])[0][0]

# Find the dominant fuzzy cluster for a query using saved cluster centers
def get_query_dominant_cluster(query_embedding):
    similarities = cosine_similarity([query_embedding], cluster_centers)[0]
    dominant_cluster = int(np.argmax(similarities))
    return dominant_cluster

# Check whether a semantically similar query already exists in cache
def check_semantic_cache(query_embedding):
    best_match = None
    best_score = 0.0

    for cached_query, cache_entry in semantic_cache.items():
        cached_embedding = cache_entry["embedding"]
        score = compute_similarity(query_embedding, cached_embedding)

        if score > best_score:
            best_score = score
            best_match = cached_query

    if best_match is not None and best_score >= SIMILARITY_THRESHOLD:
        cache_stats["hit_count"] += 1

        return {
            "cache_hit": True,
            "matched_query": best_match,
            "similarity_score": float(best_score),
            "result": semantic_cache[best_match]["result"],
            "dominant_cluster": int(semantic_cache[best_match]["cluster"])
        }

    cache_stats["miss_count"] += 1
    return {"cache_hit": False}

# Store newly computed query result in semantic cache
def store_in_cache(query, query_embedding, result_text, dominant_cluster):
    semantic_cache[query] = {
        "embedding": query_embedding,
        "result": result_text,
        "cluster": dominant_cluster
    }
    cache_stats["total_entries"] = len(semantic_cache)

# Main query processing pipeline
def process_query(query):
    # Convert input query to embedding
    query_embedding = embedding_model.encode(query)

    # First check semantic cache
    cache_result = check_semantic_cache(query_embedding)

    if cache_result["cache_hit"]:
        return {
            "query": query,
            "cache_hit": True,
            "matched_query": cache_result["matched_query"],
            "similarity_score": cache_result["similarity_score"],
            "result": cache_result["result"],
            "dominant_cluster": cache_result["dominant_cluster"]
        }

    # On cache miss, search the FAISS vector index
    query_vector = np.array([query_embedding], dtype=np.float32)
    distances, indices = index.search(query_vector, 1)

    # Retrieve the best matching document
    doc_index = int(indices[0][0])
    result_text = df.iloc[doc_index]["clean_text"]

    # Estimate dominant cluster for the query using cluster centers
    dominant_cluster = get_query_dominant_cluster(query_embedding)

    # Store fresh result in cache
    store_in_cache(query, query_embedding, result_text, dominant_cluster)

    return {
        "query": query,
        "cache_hit": False,
        "matched_query": None,
        "similarity_score": None,
        "result": result_text,
        "dominant_cluster": dominant_cluster
    }

# Return current cache statistics
def get_cache_stats():
    total_requests = cache_stats["hit_count"] + cache_stats["miss_count"]
    hit_rate = cache_stats["hit_count"] / total_requests if total_requests > 0 else 0.0

    return {
        "total_entries": cache_stats["total_entries"],
        "hit_count": cache_stats["hit_count"],
        "miss_count": cache_stats["miss_count"],
        "hit_rate": float(hit_rate)
    }

# Clear cache and reset counters
def clear_cache():
    semantic_cache.clear()
    cache_stats["total_entries"] = 0
    cache_stats["hit_count"] = 0
    cache_stats["miss_count"] = 0

# Root endpoint
@app.get("/")
def home():
    return {"message": "Trademarkia semantic search API is running"}

# Query endpoint
@app.post("/query")
def query_endpoint(request: QueryRequest):
    return process_query(request.query)

# Cache statistics endpoint
@app.get("/cache/stats")
def cache_stats_endpoint():
    return get_cache_stats()

# Cache reset endpoint
@app.delete("/cache")
def clear_cache_endpoint():
    clear_cache()
    return {"message": "Cache cleared successfully"}

Overwriting app.py


In [9]:
#53. Install pyngrok quietly.
# This library lets us create a public URL for our local FastAPI server.
!pip install pyngrok -q

In [11]:
# Import ngrok from pyngrok.
from pyngrok import ngrok

# Set your personal ngrok authentication token.
ngrok.set_auth_token("3AcF9KTMVsuLyrXaL91Q37YdVSh_4FwYPUjjpuAAHCWCvxYVb")

In [12]:
# Import ngrok again to ensure it is available in this cell.
from pyngrok import ngrok

# Kill all existing ngrok tunnels from the current session.
# This prevents conflicts or multiple active tunnels pointing to old processes.
ngrok.kill()

print("Old ngrok tunnels closed successfully.")

Old ngrok tunnels closed successfully.


In [13]:
# Create a public tunnel that forwards internet traffic
# to the FastAPI app running locally on port 8000.
public_url = ngrok.connect(8000)

# Print the public URL.
# You will use this URL to access your FastAPI app from outside the notebook.
print("Public URL:", public_url)

Public URL: NgrokTunnel: "https://arleen-unreprehensible-nontropically.ngrok-free.dev" -> "http://localhost:8000"


In [18]:
# Install FAISS for vector similarity search
!pip install faiss-cpu -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.1 MB/s eta 0:00:00


In [19]:
# Kill any existing uvicorn server processes.
# This ensures that only one clean FastAPI instance is running.
!pkill -f uvicorn

In [20]:
# Start the FastAPI application in the background using uvicorn.
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > fastapi.log 2>&1 &

In [22]:
# Display the FastAPI server log file.
!cat fastapi.log